# Notebook 05 — Classical Sentiment Baselines

## What this notebook does

This notebook builds the three classical sentiment baselines that will serve as comparison anchors for FinBERT in Step 3. The goal is not to beat FinBERT — it's to produce defensible baseline numbers so the eventual FinBERT advantage is measurable and credible.

The three baselines are:

1. **VADER** — unsupervised lexicon-based, no training required.
2. **TF-IDF + Logistic Regression** — supervised, probabilistic linear classifier.
3. **TF-IDF + Linear SVM** (calibrated) — supervised, max-margin linear classifier.

Both supervised models are trained on Financial PhraseBank (75% agreement subset), the same dataset the original FinBERT paper (Araci, 2019) used for fine-tuning. This keeps methodological choices comparable across tracks.


- Train/test split: **80/20** with internal **5-fold StratifiedKFold CV** for hyperparameter tuning (no dedicated validation split — standard for small labeled datasets).
- Evaluation at two levels in this notebook: (1) PhraseBank test split, (2) sanity check on a 10K stratified TSLA sample before full inference.
- Level 3 evaluation (manual labeling of 200–300 TSLA docs) deferred to Notebook 07.
- Models saved via joblib for reuse; full evaluation artifact saved as `eval_results.json`.

## Outputs produced

| Artifact | Path |
|---|---|
| Fitted LR pipeline | `/kaggle/working/models/lr_pipeline.joblib` |
| Fitted SVM pipeline | `/kaggle/working/models/svm_pipeline.joblib` |
| Evaluation results | `/kaggle/working/results/eval_results.json` |
| Figures (PNG) | `/kaggle/working/figures/` |
| Baseline sentiment labels on TSLA | `/kaggle/working/data/layer2_sentiment_baselines.parquet` |


## Section 0 — Setup

I start by pinning every piece of environment context I'll need later: paths, random seed, library versions. This block also doubles as a reproducibility anchor — anyone re-running this notebook six months from now can see the exact dependency versions that produced the results.


In [ ]:
# 0.1 — Install missing dependencies, then import everything
import subprocess, sys, importlib

for pkg, import_name in [("vaderSentiment", "vaderSentiment"), ("datasets", "datasets"), ("tqdm", "tqdm")]:
    if importlib.util.find_spec(import_name) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import os, json, random, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tqdm.auto import tqdm

# NLP
import spacy
import nltk
from nltk.corpus import stopwords
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# HuggingFace
from datasets import load_dataset
import datasets as hf_datasets

# Sklearn
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

# NLTK resources
for res in ["stopwords", "punkt"]:
    try:
        nltk.data.find(f"corpora/{res}" if res == "stopwords" else f"tokenizers/{res}")
    except LookupError:
        nltk.download(res, quiet=True)

# spaCy model — same one used in Notebook 04
try:
    nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
except OSError:
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=True)
    nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"All imports OK. spaCy model: {nlp.meta['name']} v{nlp.meta['version']}, pipe: {nlp.pipe_names}")

In [ ]:
# 0.2 — Configuration: seed, paths, label schema, plot style, version logging

SEED = 42
random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

# Input: Notebook 04 output, uploaded as a Kaggle dataset — replace slug if different
INPUT_DIR = Path("/kaggle/input/pfe-tsla-layer2-preprocessed")
LAYER2_PREPROCESSED_PATH = INPUT_DIR / "layer2_preprocessed.parquet"

# Output: everything under /kaggle/working/
WORK_DIR = Path("/kaggle/working")
MODELS_DIR   = WORK_DIR / "models"
RESULTS_DIR  = WORK_DIR / "results"
FIGURES_DIR  = WORK_DIR / "figures"
DATA_DIR     = WORK_DIR / "data"
for d in [MODELS_DIR, RESULTS_DIR, FIGURES_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Label schema — consistent across all models and with FinBERT output in Notebook 06
LABELS = ["negative", "neutral", "positive"]
LABEL_TO_ID = {lbl: i for i, lbl in enumerate(LABELS)}
ID_TO_LABEL = {i: lbl for lbl, i in LABEL_TO_ID.items()}

# Plot style — consistent across all figures for the thesis
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"]    = 100
plt.rcParams["savefig.dpi"]   = 200
plt.rcParams["savefig.bbox"]  = "tight"

# Version log (will be persisted to eval_results.json at the end)
import vaderSentiment as vader_pkg
VERSIONS = {
    "python":         sys.version.split()[0],
    "numpy":          np.__version__,
    "pandas":         pd.__version__,
    "sklearn":        sklearn.__version__,
    "spacy":          spacy.__version__,
    "nltk":           nltk.__version__,
    "datasets":       hf_datasets.__version__,
    "vaderSentiment": getattr(vader_pkg, "__version__", "n/a"),
    "joblib":         joblib.__version__,
}
RUN_TS = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

print(f"Seed: {SEED}  |  Run: {RUN_TS}")
print(f"Input:  {LAYER2_PREPROCESSED_PATH}")
print(f"Output: {WORK_DIR}")
print("\nLibrary versions:")
for k, v in VERSIONS.items():
    print(f"  {k:<16} {v}")

## Section 1 — Load and Explore Financial PhraseBank

The supervised baselines (Logistic Regression, SVM) need labeled training data. The TSLA dataset has no sentiment labels — documents were collected by ticker matching, not annotated. The standard academic choice is **Financial PhraseBank** (Malo et al., 2014), which is also the dataset the original FinBERT (Araci, 2019) was fine-tuned on. Reusing it here keeps comparisons across the two tracks methodologically coherent.

I use the `sentences_75agree` subset — sentences where at least 75% of 16 finance experts agreed on the label. This balances size (~3,453 sentences) and label quality. The `sentences_allagree` subset is too small and too easy; `sentences_50agree` is noisier than I'd like for academic claims.

What I want to verify in this section:
- The dataset loaded correctly and contains the expected columns.
- The class distribution is imbalanced as expected (heavy neutral skew).
- Sentence lengths are reasonable (no truncation worries for TF-IDF; this isn't a transformer).
- The label encoding matches my project-wide schema (`negative=0, neutral=1, positive=2`).


In [ ]:
# 1.1 — Load Financial PhraseBank (75% agree, from Kaggle dataset)

TXT_PATH = "/kaggle/input/datasets/ankurzing/sentiment-analysis-for-financial-news/FinancialPhraseBank/Sentences_75Agree.txt"

print(f"Loading: {TXT_PATH}")

records = []
with open(TXT_PATH, encoding="latin-1") as f:
    for line in f:
        line = line.strip()
        if not line or "@" not in line:
            continue
        text, label = line.rsplit("@", 1)   # rsplit so any @ inside the sentence is safe
        records.append({"text": text.strip(), "label": label.strip().lower()})

pb_full = pd.DataFrame(records)
pb_full["label_id"] = pb_full["label"].map(LABEL_TO_ID)

# Validate
found = set(pb_full["label"].unique())
assert found.issubset(set(LABELS)), f"Unexpected labels: {found - set(LABELS)}"

print(f"Loaded {len(pb_full):,} sentences")
print(f"Columns: {list(pb_full.columns)}")


In [ ]:
# 1.2 — Class distribution and sentence-length stats

dist = pb_full["label"].value_counts().reindex(LABELS)
dist_pct = (dist / len(pb_full) * 100).round(1)

print("Class distribution:")
for lbl in LABELS:
    print(f"  {lbl:<10} {dist[lbl]:>5,}  ({dist_pct[lbl]:>4.1f}%)")

print(f"\nTotal: {len(pb_full):,} sentences")
print(f"Imbalance ratio (majority/minority): {dist.max() / dist.min():.1f}x")

# Sentence length stats — character and word counts
pb_full["n_chars"] = pb_full["text"].str.len()
pb_full["n_words"] = pb_full["text"].str.split().str.len()

print("\nSentence length (words):")
print(pb_full["n_words"].describe().round(1).to_string())

In [ ]:
# 1.3 — Visualize class distribution and sentence-length distribution
# These plots go into the thesis "data" section.

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: class distribution (the imbalance story)
colors = {"negative": "#d62728", "neutral": "#7f7f7f", "positive": "#2ca02c"}
bars = axes[0].bar(LABELS, [dist[l] for l in LABELS], color=[colors[l] for l in LABELS])
axes[0].set_title("Financial PhraseBank — Class Distribution (75agree)", fontsize=12)
axes[0].set_ylabel("Number of sentences")
for bar, lbl in zip(bars, LABELS):
    pct = dist_pct[lbl]
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f"{int(bar.get_height()):,}\n({pct}%)", ha="center", fontsize=10)
axes[0].set_ylim(0, dist.max() * 1.15)

# Right: sentence-length distribution
axes[1].hist(pb_full["n_words"], bins=40, color="#4c72b0", edgecolor="white")
axes[1].axvline(pb_full["n_words"].median(), color="red", linestyle="--",
                label=f"median = {pb_full['n_words'].median():.0f} words")
axes[1].set_title("Sentence Length Distribution", fontsize=12)
axes[1].set_xlabel("Words per sentence")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
fig_path = FIGURES_DIR / "01_phrasebank_overview.png"
plt.savefig(fig_path)
plt.show()
print(f"Saved: {fig_path}")

In [ ]:
# 1.4 — Sample sentences per class — sanity check that labels look right

print("=" * 80)
for lbl in LABELS:
    print(f"\n--- {lbl.upper()} examples ---")
    samples = pb_full[pb_full["label"] == lbl]["text"].sample(3, random_state=SEED).tolist()
    for s in samples:
        print(f"  • {s}")
print("\n" + "=" * 80)

## Section 2 — Preprocess PhraseBank (Align with TSLA Pipeline)

This is the most important methodological step in the notebook. If PhraseBank is preprocessed differently from the TSLA documents, there's a silent distribution shift between training and inference that will quietly degrade everything downstream.

PhraseBank sentences are short, clean journalistic prose — they most closely resemble the `news` source profile from Notebook 03. So I apply the **same news-cleaning function** here, then the **same lemmatization pipeline** from Notebook 04.

Two outputs are produced per row:
- `text_clean` — for VADER (which needs raw text with capitalization and punctuation cues).
- `tokens_lemma` — for TF-IDF + LR/SVM (which need normalized lemmas).

This mirrors exactly what's in `layer2_preprocessed.parquet` for TSLA documents.


In [ ]:
# 2.1 — News cleaning function (replicated from Notebook 03)
# Operations: collapse whitespace, normalize newlines, strip leading/trailing space.
# News text needs minimal cleaning since it's already structured prose.

import re

def clean_news(text):
    """Replicates the news-source cleaning function from Notebook 03."""
    if not isinstance(text, str) or not text.strip():
        return ""
    # Collapse all whitespace (incl. newlines, tabs) into single spaces
    text = re.sub(r"\s+", " ", text)
    return text.strip()

pb_full["text_clean"] = pb_full["text"].apply(clean_news)
pb_full["clean_n_words"] = pb_full["text_clean"].str.split().str.len()

# Sanity: cleaning shouldn't have produced empty rows on PhraseBank (it's already clean)
n_empty_after = (pb_full["text_clean"].str.len() == 0).sum()
print(f"Empty rows after cleaning: {n_empty_after}")
print(f"Mean words before/after: {pb_full['n_words'].mean():.1f} / {pb_full['clean_n_words'].mean():.1f}")
print(f"\n(PhraseBank is already clean prose — minimal change expected)")


In [ ]:
# 2.2 — Lemmatization with spaCy (replicated from Notebook 04)
# Same logic: lowercase tokens, alphabetic only, drop stopwords (with financial preservations),
# protect financial domain terms from lemmatization.

# Financial preservations — kept in text despite being in NLTK stopwords
FINANCIAL_KEEP = {
    "not", "no", "nor",                     # negations — flip polarity
    "up", "down",                           # directional
    "more", "less", "very", "most",         # intensifiers
    "against", "before", "after",           # temporal/opposition
}
# News-source extra noise words (from Notebook 04)
NEWS_EXTRA_STOP = {"said", "says", "according", "read", "click", "subscribe"}

EN_STOP = (set(stopwords.words("english")) - FINANCIAL_KEEP) | NEWS_EXTRA_STOP

# Protected terms — kept as-is, never lemmatized
PROTECTED_LEMMAS = {
    "tesla", "tsla", "musk", "elon",
    "bullish", "bearish", "ipo", "etf", "sec",
    "ev", "fsd", "autopilot",
}

def lemmatize_doc(doc):
    """Take a spaCy doc → list of lemmas with project-wide rules applied."""
    tokens = []
    for tok in doc:
        if not tok.is_alpha:
            continue
        lower = tok.text.lower()
        if lower in PROTECTED_LEMMAS:
            tokens.append(lower)
            continue
        if lower in EN_STOP:
            continue
        lemma = tok.lemma_.lower()
        if len(lemma) < 2:
            continue
        tokens.append(lemma)
    return tokens

# Batch processing with spaCy nlp.pipe — much faster than row-by-row apply
print(f"Lemmatizing {len(pb_full):,} sentences with spaCy...")
texts_to_process = pb_full["text_clean"].tolist()
lemma_lists = []
for doc in tqdm(nlp.pipe(texts_to_process, batch_size=128), total=len(texts_to_process)):
    lemma_lists.append(lemmatize_doc(doc))

pb_full["tokens_lemma"] = lemma_lists
pb_full["n_tokens_lemma"] = pb_full["tokens_lemma"].str.len()

print(f"\nLemma token counts — describe:")
print(pb_full["n_tokens_lemma"].describe().round(1).to_string())

In [ ]:
# 2.3 — Spot-check: show one example per class with all three views
# Confirms the cleaning + lemmatization pipeline produces sensible output.

print("=" * 80)
for lbl in LABELS:
    row = pb_full[pb_full["label"] == lbl].sample(1, random_state=SEED + 1).iloc[0]
    print(f"\n[{lbl.upper()}]")
    print(f"  raw      : {row['text']}")
    print(f"  cleaned  : {row['text_clean']}")
    print(f"  lemmas   : {row['tokens_lemma']}")
print("\n" + "=" * 80)

# Drop rows that became empty after lemmatization (would break TF-IDF)
n_before = len(pb_full)
pb_full = pb_full[pb_full["n_tokens_lemma"] >= 1].reset_index(drop=True)
n_dropped = n_before - len(pb_full)
print(f"\nDropped {n_dropped} rows with zero lemma tokens. Final dataset size: {len(pb_full):,}")

## Section 3 — Train / Test Split

**80/20 stratified split**, no separate validation set. Hyperparameter tuning will use 5-fold StratifiedKFold CV inside the 80% training set. The 20% test set is touched exactly once at evaluation time.

This is the standard NLP convention for small labeled datasets (<5K examples). It maximizes the test set size — which gives tighter bootstrap CIs on the per-class F1 numbers we'll need to defend in the thesis.


In [ ]:
# 3.1 — Stratified 80/20 split

X = pb_full["tokens_lemma"].tolist()        # token lists for LR/SVM
X_text = pb_full["text_clean"].tolist()     # cleaned strings for VADER
y = pb_full["label"].tolist()               # string labels (negative/neutral/positive)

idx = np.arange(len(pb_full))

idx_train, idx_test = train_test_split(
    idx, test_size=0.20, random_state=SEED, stratify=y
)

X_train       = [X[i] for i in idx_train]
X_train_text  = [X_text[i] for i in idx_train]
y_train       = [y[i] for i in idx_train]

X_test        = [X[i] for i in idx_test]
X_test_text   = [X_text[i] for i in idx_test]
y_test        = [y[i] for i in idx_test]

print(f"Train: {len(X_train):,} ({len(X_train)/len(pb_full)*100:.1f}%)")
print(f"Test : {len(X_test):,} ({len(X_test)/len(pb_full)*100:.1f}%)")

# Verify stratification preserved class proportions in both splits
def class_dist(labels):
    s = pd.Series(labels).value_counts()
    return {l: f"{s.get(l, 0)} ({s.get(l, 0)/len(labels)*100:.1f}%)" for l in LABELS}

print("\nTrain class distribution:")
for k, v in class_dist(y_train).items():
    print(f"  {k:<10} {v}")
print("\nTest class distribution:")
for k, v in class_dist(y_test).items():
    print(f"  {k:<10} {v}")

# Stash the test set indices for reproducibility
TEST_INDICES = idx_test.tolist()

## Section 4 — VADER Baseline (Unsupervised)

VADER (Valence Aware Dictionary and sEntiment Reasoner; Hutto & Gilbert, 2014) is a rule-based system built on a hand-curated lexicon of ~7,500 words and emoticons. It's not trained — there's nothing to fit. I apply it directly to the PhraseBank test set using the cleaned **raw text** (not lemmas), because VADER's heuristics rely on punctuation and capitalization cues that lemmatization would destroy.

Decision rule (Hutto & Gilbert convention):
- `compound >= +0.05` → positive
- `compound <= -0.05` → negative
- otherwise → neutral

**Expected weakness — this is the academic point.** VADER's lexicon is general-purpose. It has no entry for `bullish`, `bearish`, `beat estimates`, `miss guidance`, `dilution`, etc. On a sentence like *"Tesla missed Q3 delivery estimates"* it will likely return neutral — exactly the failure mode that motivates FinBERT.


In [ ]:
# 4.1 — Apply VADER to PhraseBank test set

vader = SentimentIntensityAnalyzer()

def vader_predict(text):
    """Return (label, compound, pos, neg, neu) for a single text."""
    s = vader.polarity_scores(text)
    c = s["compound"]
    if c >= 0.05:
        label = "positive"
    elif c <= -0.05:
        label = "negative"
    else:
        label = "neutral"
    return label, c, s["pos"], s["neg"], s["neu"]

vader_test_records = [vader_predict(t) for t in tqdm(X_test_text, desc="VADER on test set")]
vader_test_df = pd.DataFrame(vader_test_records, columns=["pred", "compound", "pos", "neg", "neu"])
y_pred_vader = vader_test_df["pred"].tolist()

# Quick sanity print
print("\nVADER predictions on test set — distribution:")
print(pd.Series(y_pred_vader).value_counts().reindex(LABELS).to_string())

## Section 5 — TF-IDF + Logistic Regression

This is the first supervised baseline. I use a `Pipeline` with `TfidfVectorizer` followed by `LogisticRegression`, then wrap the whole thing in `GridSearchCV` for hyperparameter tuning via 5-fold StratifiedKFold CV.

**Critical methodology:** the vectorizer is fitted **only on training folds**, never on test data. The Pipeline + GridSearchCV combination enforces this automatically — there's no risk of vocabulary leakage.

**Hyperparameters tuned:**
- `tfidf__ngram_range` — `(1,1)` unigrams only vs `(1,2)` unigrams + bigrams. Bigrams capture critical financial phrases like *"not bullish"*, *"beat estimates"*, *"short squeeze"*.
- `clf__C` — inverse L2 regularization strength, swept over `{0.01, 0.1, 1, 10}`.

**Fixed:** `class_weight='balanced'` (mandatory given 59% neutral imbalance) and `analyzer=lambda x: x` because inputs are already token lists from the lemmatization step.


In [ ]:
# 5.1 — Define the LR pipeline and grid

# Identity analyzer — inputs are already token lists, skip TfidfVectorizer's internal tokenization
def identity_tokenizer(x):
    return x

pipe_lr = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer=identity_tokenizer,
        lowercase=False,
        min_df=5,
        max_df=0.95,
        sublinear_tf=True,
        max_features=50_000,
    )),
    ("clf", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="liblinear",
        random_state=SEED,
    )),
])

grid_lr = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "clf__C":             [0.01, 0.1, 1.0, 10.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

gs_lr = GridSearchCV(
    pipe_lr, grid_lr, cv=cv, scoring="f1_macro",
    n_jobs=-1, verbose=1, return_train_score=True,
)

print(f"Fitting GridSearchCV — {len(grid_lr['tfidf__ngram_range']) * len(grid_lr['clf__C'])} configs × 5 folds = "
      f"{len(grid_lr['tfidf__ngram_range']) * len(grid_lr['clf__C']) * 5} fits")
gs_lr.fit(X_train, y_train)

print(f"\nBest LR params: {gs_lr.best_params_}")
print(f"Best CV macro-F1: {gs_lr.best_score_:.4f}")

best_lr = gs_lr.best_estimator_

In [ ]:
# 5.2 — Evaluate best LR on the held-out test set

y_pred_lr = best_lr.predict(X_test)
y_proba_lr = best_lr.predict_proba(X_test)  # class order = best_lr.classes_

print("Logistic Regression — Test Set Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred_lr, labels=LABELS, digits=4))

## Section 6 — TF-IDF + Linear SVM (Calibrated)

Second supervised baseline. Same TF-IDF feature space as LR, but using **`LinearSVC`** which optimizes hinge loss instead of cross-entropy. Linear SVM has historically been the strongest non-neural baseline for sparse text classification (Joachims, 1998).

**Two crucial implementation details:**

1. **`LinearSVC` not `SVC(kernel='linear')`** — `LinearSVC` uses the `liblinear` solver which scales linearly with sample count on sparse data. `SVC` is O(n²) and would be impractical even at this small scale.

2. **`CalibratedClassifierCV` wrap** — `LinearSVC` doesn't expose `predict_proba` by default (it only returns decision scores). I wrap it with Platt scaling (`method='sigmoid'`) so SVM produces valid probabilities — necessary for downstream sentiment aggregation in Step 4 and for fair comparison with LR/FinBERT outputs.


In [ ]:
# 6.1 — Define the SVM pipeline (with probability calibration) and grid

base_svm = LinearSVC(
    class_weight="balanced",
    max_iter=2000,
    random_state=SEED,
    dual="auto",
)
svm_with_proba = CalibratedClassifierCV(base_svm, method="sigmoid", cv=5)

pipe_svm = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer=identity_tokenizer,
        lowercase=False,
        min_df=5,
        max_df=0.95,
        sublinear_tf=True,
        max_features=50_000,
    )),
    ("clf", svm_with_proba),
])

# Note: when nesting CalibratedClassifierCV around LinearSVC, the C parameter is reached via
# clf__estimator__C (sklearn >= 1.2 uses 'estimator', older versions used 'base_estimator')
grid_svm = {
    "tfidf__ngram_range":   [(1, 1), (1, 2)],
    "clf__estimator__C":    [0.01, 0.1, 1.0, 10.0],
}

gs_svm = GridSearchCV(
    pipe_svm, grid_svm, cv=cv, scoring="f1_macro",
    n_jobs=-1, verbose=1, return_train_score=True,
)

print(f"Fitting GridSearchCV — {len(grid_svm['tfidf__ngram_range']) * len(grid_svm['clf__estimator__C'])} configs × 5 folds")
gs_svm.fit(X_train, y_train)

print(f"\nBest SVM params: {gs_svm.best_params_}")
print(f"Best CV macro-F1: {gs_svm.best_score_:.4f}")

best_svm = gs_svm.best_estimator_

In [ ]:
# 6.2 — Evaluate best SVM on the held-out test set

y_pred_svm = best_svm.predict(X_test)
y_proba_svm = best_svm.predict_proba(X_test)

print("Linear SVM (calibrated) — Test Set Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred_svm, labels=LABELS, digits=4))

## Section 7 — PhraseBank Test Set Comparison

This is the headline comparison table. Three models, one test set, identical evaluation protocol. The metrics:

- **Accuracy** — overall correctness. Reported but not the primary metric (PhraseBank is imbalanced).
- **Macro-F1** — primary metric. Averages F1 across classes equally, so the small negative class isn't drowned out.
- **Per-class precision / recall / F1** — to spot which models systematically miss certain classes.
- **Confusion matrices** — to see *where* errors happen, not just how many.

Visualizations: confusion-matrix heatmaps (3 side-by-side, normalized) + per-class F1 grouped bar chart.


In [ ]:
# 7.1 — Compute summary metrics for all three models, in one table

def compute_metrics(y_true, y_pred, model_name):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, average=None, zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=LABELS)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", labels=LABELS)
    acc = accuracy_score(y_true, y_pred)
    return {
        "model": model_name,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        **{f"P_{lbl}": p[i] for i, lbl in enumerate(LABELS)},
        **{f"R_{lbl}": r[i] for i, lbl in enumerate(LABELS)},
        **{f"F1_{lbl}": f[i] for i, lbl in enumerate(LABELS)},
    }

metric_rows = [
    compute_metrics(y_test, y_pred_vader, "VADER"),
    compute_metrics(y_test, y_pred_lr,    "TF-IDF + LR"),
    compute_metrics(y_test, y_pred_svm,   "TF-IDF + SVM"),
]
metrics_df = pd.DataFrame(metric_rows).set_index("model")

# Display key columns first, then per-class detail
key_cols = ["accuracy", "macro_f1", "weighted_f1"]
detail_cols = [c for c in metrics_df.columns if c not in key_cols]

print("Headline metrics:")
print(metrics_df[key_cols].round(4).to_string())

print("\nPer-class detail:")
print(metrics_df[detail_cols].round(4).to_string())

In [ ]:
# 7.2 — Confusion matrix heatmaps (3 side-by-side, normalized by true class)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
predictions_by_model = [
    ("VADER",        y_pred_vader),
    ("TF-IDF + LR",  y_pred_lr),
    ("TF-IDF + SVM", y_pred_svm),
]

for ax, (name, y_pred) in zip(axes, predictions_by_model):
    cm = confusion_matrix(y_test, y_pred, labels=LABELS, normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", cbar=False,
                xticklabels=LABELS, yticklabels=LABELS, ax=ax,
                vmin=0, vmax=1, annot_kws={"fontsize": 11})
    ax.set_title(f"{name}\nmacro-F1 = {f1_score(y_test, y_pred, average='macro'):.3f}", fontsize=11)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True" if name == "VADER" else "")

plt.suptitle("Confusion Matrices on PhraseBank Test Set (row-normalized)", fontsize=13, y=1.05)
plt.tight_layout()
fig_path = FIGURES_DIR / "02_confusion_matrices.png"
plt.savefig(fig_path)
plt.show()
print(f"Saved: {fig_path}")

In [ ]:
# 7.3 — Per-class F1 grouped bar chart

f1_per_class = pd.DataFrame({
    name: [f1_score(y_test, y_pred, labels=[lbl], average=None, zero_division=0)[0] for lbl in LABELS]
    for name, y_pred in predictions_by_model
}, index=LABELS)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(LABELS))
width = 0.27
colors = {"VADER": "#9467bd", "TF-IDF + LR": "#ff7f0e", "TF-IDF + SVM": "#1f77b4"}

for i, name in enumerate(f1_per_class.columns):
    bars = ax.bar(x + (i - 1) * width, f1_per_class[name], width, label=name, color=colors[name])
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{bar.get_height():.2f}", ha="center", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(LABELS)
ax.set_ylabel("F1 score")
ax.set_ylim(0, 1.05)
ax.set_title("Per-Class F1 on PhraseBank Test Set", fontsize=12)
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig_path = FIGURES_DIR / "03_per_class_f1.png"
plt.savefig(fig_path)
plt.show()
print(f"Saved: {fig_path}")

In [ ]:
# 7.4 — Hyperparameter sweep plot (C vs. mean CV macro-F1) for LR and SVM
# Useful for the thesis defense: shows that the chosen C wasn't a lucky pick.

def sweep_data(grid_search, c_param_name):
    """Pull (ngram_range, C, mean_cv_f1, std_cv_f1) tuples from cv_results_."""
    res = grid_search.cv_results_
    rows = []
    for i in range(len(res["params"])):
        p = res["params"][i]
        rows.append({
            "ngram": str(p["tfidf__ngram_range"]),
            "C": p[c_param_name],
            "mean_f1": res["mean_test_score"][i],
            "std_f1": res["std_test_score"][i],
        })
    return pd.DataFrame(rows)

sweep_lr  = sweep_data(gs_lr,  "clf__C")
sweep_svm = sweep_data(gs_svm, "clf__estimator__C")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

for ax, (name, sw) in zip(axes, [("Logistic Regression", sweep_lr), ("Linear SVM", sweep_svm)]):
    for ngram, group in sw.groupby("ngram"):
        group = group.sort_values("C")
        ax.errorbar(group["C"], group["mean_f1"], yerr=group["std_f1"],
                    marker="o", capsize=4, label=f"ngram={ngram}", linewidth=2)
    ax.set_xscale("log")
    ax.set_xlabel("C (inverse regularization)")
    ax.set_title(name)
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Mean CV macro-F1 (± std)")
plt.suptitle("Hyperparameter Sweep — 5-fold CV macro-F1 vs C", fontsize=12, y=1.02)
plt.tight_layout()
fig_path = FIGURES_DIR / "04_hyperparam_sweep.png"
plt.savefig(fig_path)
plt.show()
print(f"Saved: {fig_path}")

## Section 8 — Top-N Words per Class (Interpretability Artifact)

Linear models give us something FinBERT can't directly: a transparent list of which words drive each class prediction. I extract the top-20 features (highest positive coefficients) per class for both LR and SVM.

This is a concrete deliverable for the thesis — a table reviewers can read immediately to verify the models learned something semantically sensible (e.g. *surge*, *upgrade*, *beat* under "positive"; *miss*, *cut*, *lawsuit* under "negative"). FinBERT's analogous interpretation requires LIME/SHAP/attention analysis, which is more involved.


In [ ]:
# 8.1 — Top-20 words per class for LR and SVM

def top_words_per_class(pipeline, top_n=20):
    """Extract top-N features per class from a fitted Pipeline (TF-IDF + linear classifier)."""
    vec = pipeline.named_steps["tfidf"]
    clf_step = pipeline.named_steps["clf"]
    feature_names = np.array(vec.get_feature_names_out())

    # CalibratedClassifierCV stores fitted estimators; average their coefficients
    if hasattr(clf_step, "calibrated_classifiers_"):
        coefs = np.mean([cc.estimator.coef_ for cc in clf_step.calibrated_classifiers_], axis=0)
        classes = clf_step.classes_
    else:
        coefs = clf_step.coef_
        classes = clf_step.classes_

    out = {}
    for i, cls in enumerate(classes):
        top_idx = np.argsort(coefs[i])[::-1][:top_n]
        out[cls] = list(zip(feature_names[top_idx], coefs[i, top_idx].round(3)))
    return out

top_lr  = top_words_per_class(best_lr,  top_n=20)
top_svm = top_words_per_class(best_svm, top_n=20)

print("=" * 70)
print("LOGISTIC REGRESSION — Top 20 features per class")
print("=" * 70)
for cls in LABELS:
    print(f"\n[{cls.upper()}]")
    for word, weight in top_lr[cls]:
        print(f"  {word:<30s} {weight:+.3f}")

print("\n" + "=" * 70)
print("LINEAR SVM — Top 20 features per class")
print("=" * 70)
for cls in LABELS:
    print(f"\n[{cls.upper()}]")
    for word, weight in top_svm[cls]:
        print(f"  {word:<30s} {weight:+.3f}")

## Section 9 — Apply to TSLA Data — Phase 1: 10K Sanity Check

Before running all three models on the full ~87K TSLA dataset, I do a stratified 10K sample first. Reasons:

1. **Catch bugs cheap.** If TF-IDF complains about empty token lists, or VADER chokes on a weird character, I want to know now — not 60 minutes into the full run.
2. **Sanity-check label distributions per source.** I expect different distributions per source: news should be more neutral, twitter_musk more polarized. If results don't roughly match this expectation, something's wrong upstream.
3. **Visual spot-check.** I print 5 random predictions per source × per model to eyeball that nothing obvious is off.

Only after this passes do I run the full ~87K with checkpointing in Section 10.


In [ ]:
# 9.1 — Load layer2_preprocessed.parquet and prepare TSLA inputs

print(f"Loading {LAYER2_PREPROCESSED_PATH}...")
tsla = pd.read_parquet("/kaggle/input/datasets/leev75/processd/layer2_preprocced")
print(f"Loaded {len(tsla):,} TSLA documents")
print(f"Columns: {list(tsla.columns)}")
print(f"\nSource distribution:")
print(tsla["source"].value_counts().to_string())

# tokens_lemma was JSON-serialized in Notebook 04 — deserialize for sklearn
if isinstance(tsla["tokens_lemma"].iloc[0], str):
    print("\nDeserializing tokens_lemma from JSON strings...")
    tsla["tokens_lemma"] = tsla["tokens_lemma"].apply(json.loads)

# Drop rows with empty token lists (would break TF-IDF) and empty cleaned text
tsla["n_tokens_lemma"] = tsla["tokens_lemma"].str.len()
n_before = len(tsla)
tsla = tsla[(tsla["n_tokens_lemma"] >= 1) & (tsla["text"].str.strip() != "")].reset_index(drop=True)
print(f"\nDropped {n_before - len(tsla)} rows with empty tokens or text")
print(f"Final TSLA dataset for inference: {len(tsla):,}")

In [ ]:
# 9.2 — Stratified 10K sample (~2.5K per source)

SAMPLE_PER_SOURCE = 2_500

tsla_sample = (
    tsla.groupby("source", group_keys=False)
        .apply(lambda g: g.sample(min(SAMPLE_PER_SOURCE, len(g)), random_state=SEED))
        .reset_index(drop=True)
)
print(f"Sample size: {len(tsla_sample):,}")
print("\nSample per source:")
print(tsla_sample["source"].value_counts().to_string())

In [ ]:
# 9.3 — Apply all three models to the 10K sample

# VADER — on cleaned raw text
print("Running VADER on sample...")
vader_records = [vader_predict(t) for t in tqdm(tsla_sample["text"].tolist(), desc="VADER")]
vader_sample_df = pd.DataFrame(vader_records, columns=["vader_label", "vader_compound", "vader_pos", "vader_neg", "vader_neu"])

# LR — on lemmatized tokens
print("\nRunning LR on sample...")
lr_pred  = best_lr.predict(tsla_sample["tokens_lemma"].tolist())
lr_proba = best_lr.predict_proba(tsla_sample["tokens_lemma"].tolist())
lr_class_order = list(best_lr.classes_)
lr_sample_df = pd.DataFrame({
    "lr_label":      lr_pred,
    "lr_proba_neg":  lr_proba[:, lr_class_order.index("negative")],
    "lr_proba_neu":  lr_proba[:, lr_class_order.index("neutral")],
    "lr_proba_pos":  lr_proba[:, lr_class_order.index("positive")],
})

# SVM — on lemmatized tokens
print("\nRunning SVM on sample...")
svm_pred  = best_svm.predict(tsla_sample["tokens_lemma"].tolist())
svm_proba = best_svm.predict_proba(tsla_sample["tokens_lemma"].tolist())
svm_class_order = list(best_svm.classes_)
svm_sample_df = pd.DataFrame({
    "svm_label":      svm_pred,
    "svm_proba_neg":  svm_proba[:, svm_class_order.index("negative")],
    "svm_proba_neu":  svm_proba[:, svm_class_order.index("neutral")],
    "svm_proba_pos":  svm_proba[:, svm_class_order.index("positive")],
})

tsla_sample_pred = pd.concat(
    [tsla_sample.reset_index(drop=True), vader_sample_df, lr_sample_df, svm_sample_df],
    axis=1,
)
print(f"\nDone. Sample with predictions: {tsla_sample_pred.shape}")

In [ ]:
# 9.4 — Sanity check: label distribution by source × model

models = ["vader_label", "lr_label", "svm_label"]
sources = sorted(tsla_sample_pred["source"].unique())

dist_records = []
for src in sources:
    sub = tsla_sample_pred[tsla_sample_pred["source"] == src]
    for m in models:
        for lbl in LABELS:
            dist_records.append({
                "source": src,
                "model":  m.replace("_label", ""),
                "label":  lbl,
                "pct":    (sub[m] == lbl).mean() * 100,
            })
dist_long = pd.DataFrame(dist_records)

# Pivot for readable table
table = dist_long.pivot_table(index=["source", "model"], columns="label", values="pct").round(1)
table = table[LABELS]
print("Label distribution (%) by source × model — sanity check:")
print(table.to_string())

In [ ]:
# 9.5 — Visualize sanity-check distributions

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
colors_lbl = {"negative": "#d62728", "neutral": "#7f7f7f", "positive": "#2ca02c"}

for ax, model in zip(axes, ["vader", "lr", "svm"]):
    sub = dist_long[dist_long["model"] == model]
    pivoted = sub.pivot(index="source", columns="label", values="pct")[LABELS]
    pivoted.plot(kind="bar", stacked=True, ax=ax,
                 color=[colors_lbl[l] for l in LABELS], edgecolor="white", width=0.7)
    ax.set_title(f"{model.upper()} — sentiment distribution by source")
    ax.set_xlabel("")
    ax.set_ylabel("% of documents" if model == "vader" else "")
    ax.set_ylim(0, 100)
    ax.legend(title="label", loc="upper right", fontsize=9)
    ax.tick_params(axis="x", rotation=30)

plt.suptitle("Sanity check on 10K sample: do label distributions look reasonable per source?",
             fontsize=12, y=1.03)
plt.tight_layout()
fig_path = FIGURES_DIR / "05_sanity_check_distributions.png"
plt.savefig(fig_path)
plt.show()
print(f"Saved: {fig_path}")

In [ ]:
# 9.6 — Visual spot-check: 3 random predictions per source per model

print("Visual spot-check — does each model's output look plausible?\n")
for src in sources:
    print("=" * 80)
    print(f"SOURCE: {src}")
    print("=" * 80)
    sub = tsla_sample_pred[tsla_sample_pred["source"] == src].sample(3, random_state=SEED)
    for _, row in sub.iterrows():
        text = row["text"][:200] + ("..." if len(row["text"]) > 200 else "")
        print(f"\n  TEXT: {text}")
        print(f"    VADER: {row['vader_label']:<10s} (compound={row['vader_compound']:+.2f})")
        print(f"    LR   : {row['lr_label']:<10s} (P={row[f'lr_proba_{row["lr_label"][:3]}']:.2f})")
        print(f"    SVM  : {row['svm_label']:<10s} (P={row[f'svm_proba_{row["svm_label"][:3]}']:.2f})")
    print()

## Section 10 — Full TSLA Inference (with Checkpointing)

Sanity check passed. Now run all three models on the full ~87K rows. To survive Kaggle kernel timeouts and to be able to resume on failure, I write intermediate Parquet checkpoints every 10K rows. The final consolidated output is `layer2_sentiment_baselines.parquet`.

**Schema** (preserves all original Layer 2 columns + appends sentiment columns):

| Added column | Source |
|---|---|
| `vader_label`, `vader_compound`, `vader_pos`, `vader_neg`, `vader_neu` | VADER |
| `lr_label`, `lr_proba_neg`, `lr_proba_neu`, `lr_proba_pos` | LR |
| `svm_label`, `svm_proba_neg`, `svm_proba_neu`, `svm_proba_pos` | SVM |


In [ ]:
# 10.1 — Apply all three models to the full TSLA dataset, with checkpointing

CHECKPOINT_DIR = WORK_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHUNK_SIZE = 10_000

def predict_chunk(chunk):
    """Run all three models on a chunk and return a DataFrame of predictions."""
    texts  = chunk["text"].tolist()
    tokens = chunk["tokens_lemma"].tolist()

    # VADER
    v = [vader_predict(t) for t in texts]
    v_df = pd.DataFrame(v, columns=["vader_label", "vader_compound", "vader_pos", "vader_neg", "vader_neu"])

    # LR
    lr_p  = best_lr.predict(tokens)
    lr_pr = best_lr.predict_proba(tokens)
    lr_co = list(best_lr.classes_)
    lr_df = pd.DataFrame({
        "lr_label":     lr_p,
        "lr_proba_neg": lr_pr[:, lr_co.index("negative")],
        "lr_proba_neu": lr_pr[:, lr_co.index("neutral")],
        "lr_proba_pos": lr_pr[:, lr_co.index("positive")],
    })

    # SVM
    sv_p  = best_svm.predict(tokens)
    sv_pr = best_svm.predict_proba(tokens)
    sv_co = list(best_svm.classes_)
    sv_df = pd.DataFrame({
        "svm_label":     sv_p,
        "svm_proba_neg": sv_pr[:, sv_co.index("negative")],
        "svm_proba_neu": sv_pr[:, sv_co.index("neutral")],
        "svm_proba_pos": sv_pr[:, sv_co.index("positive")],
    })

    return pd.concat([chunk.reset_index(drop=True), v_df, lr_df, sv_df], axis=1)

# Process in chunks with checkpointing
n_chunks = (len(tsla) + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"Processing {len(tsla):,} rows in {n_chunks} chunks of {CHUNK_SIZE:,}")

chunk_paths = []
for i in tqdm(range(n_chunks), desc="Chunks"):
    ckpt_path = CHECKPOINT_DIR / f"sentiment_chunk_{i:03d}.parquet"
    if ckpt_path.exists():
        chunk_paths.append(ckpt_path)
        continue
    start, end = i * CHUNK_SIZE, min((i + 1) * CHUNK_SIZE, len(tsla))
    chunk = tsla.iloc[start:end].copy()
    chunk_pred = predict_chunk(chunk)
    chunk_pred.to_parquet(ckpt_path, index=False)
    chunk_paths.append(ckpt_path)

print(f"\nAll {n_chunks} chunks processed. Concatenating...")
tsla_with_sentiment = pd.concat([pd.read_parquet(p) for p in chunk_paths], ignore_index=True)
print(f"Final dataset shape: {tsla_with_sentiment.shape}")

In [ ]:
# 10.2 — Save the final layer2_sentiment_baselines.parquet

OUTPUT_PATH = DATA_DIR / "layer2_sentiment_baselines.parquet"
tsla_with_sentiment.to_parquet(OUTPUT_PATH, index=False)

print(f"Saved: {OUTPUT_PATH}")
print(f"Rows:  {len(tsla_with_sentiment):,}")
print(f"Cols:  {len(tsla_with_sentiment.columns)}")

# Quick distribution summary on the full dataset
print("\nFull-dataset label distribution (%) per model:")
for m in ["vader_label", "lr_label", "svm_label"]:
    s = tsla_with_sentiment[m].value_counts(normalize=True).reindex(LABELS) * 100
    print(f"  {m.replace('_label', '').upper():<6}  " + "  ".join(f"{l}={s[l]:.1f}%" for l in LABELS))

In [ ]:
# 10.3 — Save fitted models and the consolidated evaluation results JSON

# Models
lr_model_path  = MODELS_DIR / "lr_pipeline.joblib"
svm_model_path = MODELS_DIR / "svm_pipeline.joblib"
joblib.dump(best_lr,  lr_model_path)
joblib.dump(best_svm, svm_model_path)
print(f"Saved: {lr_model_path}")
print(f"Saved: {svm_model_path}")

# Consolidated eval_results.json
def to_jsonable(obj):
    """Convert numpy scalars/arrays to plain Python types for JSON serialization."""
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    return obj

eval_results = {
    "run_timestamp": RUN_TS,
    "seed": SEED,
    "library_versions": VERSIONS,
    "phrasebank": {
        "subset": "sentences_75agree",
        "total_rows": len(pb_full),
        "train_rows": len(X_train),
        "test_rows":  len(X_test),
        "class_distribution_pct": {l: float((pd.Series(y).value_counts(normalize=True) * 100).get(l, 0)) for l in LABELS},
    },
    "models": {
        "vader": {
            "type": "lexicon_unsupervised",
            "test_metrics": metrics_df.loc["VADER"].to_dict(),
        },
        "logistic_regression": {
            "type": "tfidf_lr",
            "best_params": gs_lr.best_params_,
            "best_cv_macro_f1": gs_lr.best_score_,
            "test_metrics": metrics_df.loc["TF-IDF + LR"].to_dict(),
            "cv_results": sweep_lr.to_dict(orient="records"),
            "model_path": str(lr_model_path),
        },
        "linear_svm": {
            "type": "tfidf_linearsvc_calibrated",
            "best_params": gs_svm.best_params_,
            "best_cv_macro_f1": gs_svm.best_score_,
            "test_metrics": metrics_df.loc["TF-IDF + SVM"].to_dict(),
            "cv_results": sweep_svm.to_dict(orient="records"),
            "model_path": str(svm_model_path),
        },
    },
    "tsla_inference": {
        "input_path":  str(LAYER2_PREPROCESSED_PATH),
        "output_path": str(OUTPUT_PATH),
        "n_documents": len(tsla_with_sentiment),
        "label_distribution_pct": {
            m.replace("_label", ""): {
                l: float((tsla_with_sentiment[m] == l).mean() * 100) for l in LABELS
            }
            for m in ["vader_label", "lr_label", "svm_label"]
        },
    },
    "figures": [str(p) for p in sorted(FIGURES_DIR.glob("*.png"))],
}

results_path = RESULTS_DIR / "eval_results.json"
with open(results_path, "w") as f:
    json.dump(to_jsonable(eval_results), f, indent=2, default=str)

print(f"\nSaved: {results_path}")

## Section 11 — Summary, Limitations, Bridge to Notebook 06

### What was produced

- Three sentiment baselines fitted, evaluated, and applied to all ~87K TSLA documents.
- Five thesis-quality figures saved to `/kaggle/working/figures/`.
- Consolidated `eval_results.json` with every metric, hyperparameter choice, and CV score.
- Two fitted pipelines saved via joblib for reuse in Notebook 07.
- Final dataset `layer2_sentiment_baselines.parquet` ready as input to the FinBERT comparison notebook.

### Honest limitations to document in the thesis

1. **PhraseBank is not TSLA.** Both supervised models were trained on general financial news prose. TSLA documents — especially Twitter/Reddit — are stylistically different. Performance on PhraseBank test is an upper bound; out-of-domain TSLA performance will be measured properly in Notebook 07 against manually labeled data.
2. **VADER's lexicon is general-purpose**, not financial. The expected weakness on financial-specific phrasing (*beat estimates*, *bullish*, *miss guidance*) is the academic motivation for FinBERT, and will show up clearly in the Notebook 07 comparison.
3. **`class_weight='balanced'` is a soft fix for a hard problem.** PhraseBank is 59% neutral; the balancing helps but doesn't fully eliminate the model's preference for the majority class. Reweighting in downstream sentiment aggregation (Step 4) may be needed.
4. **CalibratedClassifierCV adds ~5x training time** but produces SVM probabilities comparable to LR/FinBERT outputs. Worth the cost.

### Next steps

- **Notebook 06** — `06_finbert_sentiment.ipynb`: Apply ProsusAI/FinBERT to the full TSLA dataset. Save `layer2_sentiment_finbert.parquet`. Same schema convention as this notebook (`finbert_label`, `finbert_proba_*`).
- **Notebook 07** — `07_sentiment_comparison.ipynb`: Merge baseline + FinBERT outputs. Produce manually labeled 200–300 TSLA sample. Compute Cohen's kappa for inter-model agreement. Bootstrap CIs for the headline macro-F1 differences. This is the notebook that produces the master sentiment-model comparison for the thesis.
- **Then Step 4 (Feature Engineering)**: Aggregate per-document sentiment into daily indices weighted by Layer 1 engagement metrics; join with yfinance OHLCV; build the final feature matrix for Step 5.
